In [1]:
# 🔷 Question 2: Design a Multi-Agent Workflow with LangGraph (25 Marks)
# 🧩 Scenario
# You are building an AI-powered customer support system for a fintech company.

# The system must handle:

# Transaction queries
# Fraud detection flags
# Refund requests
# General FAQs
# The company wants:

# High accuracy
# Step-by-step reasoning
# Ability to retry if answer is incorrect
# Modular, scalable architecture
# 💻 Task
# Design and implement a multi-agent workflow using LangGraph (or similar framework).

# ✅ 1. Agent Design
# Define at least 3 agents, such as:

# Retrieval Agent
# Reasoning/Answer Agent
# Validation Agent
# Explain briefly (in comments or code):

# Each agent’s role
# Input/output
# ✅ 2. Graph Workflow Implementation
# Write code or pseudo-code to:

# Define state
# Add nodes (agents)
# Define edges
# Implement conditional logic
# 👉 Must include:

# Retry loop if validation fails
# Clear start and end states
# ✅ 3. State Management
# Show how state evolves across steps:

# Query
# Context
# Intermediate reasoning
# Final answer
# Validation flag
# ✅ 4. Task Delegation & Communication
# Demonstrate:

# How agents pass information
# How decisions are made between agents
# 🎯 Expected Outcome
# A clear multi-step, graph-based agent system that:

# Handles complex queries
# Demonstrates reasoning + validation
# Uses proper orchestration 

In [2]:
# 3main agents are :-

# retrieval agent
# Reasoning Agent
# Validation Agent

In [14]:
# Retrieves relevant data from knowledge base
def retrieval_agent(state):
    
    state["retries"] += 1
    
    context = f"Retry info for: {state['query']}"
    
    return {**state, "context": context}

In [4]:
# Generates answer using reasoning
def reasoning_agent(state):
    
    query = state["query"]
    context = state["context"]
    
    answer = f"""
    Step 1: Understand query → {query}
    Step 2: Use context → {context}
    Final Answer: Based on records, your issue is resolved.
    """
    
    return {**state, "answer": answer}

In [5]:
# Validates answer quality
def validation_agent(state):
    
    answer = state["answer"]
    
    if "resolved" in answer.lower():
        return {**state, "valid": True}
    
    return {**state, "valid": False}

In [6]:
from typing import TypedDict

class AgentState(TypedDict):
    query: str
    context: str
    answer: str
    valid: bool
    retries: int

In [7]:
from langgraph.graph import StateGraph

graph = StateGraph(AgentState)

In [8]:
graph.add_node("retrieval", retrieval_agent)
graph.add_node("reasoning", reasoning_agent)
graph.add_node("validation", validation_agent)

In [9]:
graph.set_entry_point("retrieval")

graph.add_edge("retrieval", "reasoning")
graph.add_edge("reasoning", "validation")

In [10]:
def check_validation(state):
    
    if state["valid"] == True:
        return "end"
    
    elif state["retries"] < 2:
        return "retry"
    
    else:
        return "end"

In [11]:
graph.add_conditional_edges(
    "validation",
    check_validation,
    {
        "retry": "retrieval",   # 🔁 retry loop
        "end": "__end__"
    }
)

In [12]:
app = graph.compile()

In [13]:
state = {
    "query": "My payment failed, what should I do?",
    "context": "",
    "answer": "",
    "valid": False,
    "retries": 0
}

In [17]:
result = app.invoke(state)

print(result)

{'query': 'My payment failed, what should I do?', 'context': 'Relevant info for: My payment failed, what should I do?', 'answer': '\n    Step 1: Understand query → My payment failed, what should I do?\n    Step 2: Use context → Relevant info for: My payment failed, what should I do?\n    Final Answer: Based on records, your issue is resolved.\n    ', 'valid': True, 'retries': 0}


In [18]:
print(result["answer"])


    Step 1: Understand query → My payment failed, what should I do?
    Step 2: Use context → Relevant info for: My payment failed, what should I do?
    Final Answer: Based on records, your issue is resolved.
    
